# 01 - Entrainement ModCloth Fit Model V3 experimental

**But :** preparer, analyser, entrainer et sauvegarder les experiences ModCloth V3 pour la prediction de fit (`small`, `fit`, `large`).

Ce notebook execute uniquement le pipeline ModCloth tabulaire :
- telechargement Kaggle ;
- inspection et analyses descriptives ;
- entrainement V3 experimental ;
- comparaison `all` vs categories explicites ;
- sauvegarde des artefacts dans Google Drive.

Il ne traite ni Fashion Product Images Small, ni Polyvore.

> **Important - promotion**
>
> Les artefacts V3 restent experimentaux. Ne copie rien vers `models/fit_active/` depuis ce notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Monter Google Drive


In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=True)

DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"
print("Drive root:", DRIVE_ROOT)


## 2. Cloner ou mettre à jour le repo GitHub

Le code du projet reste dans GitHub. Colab clone une copie temporaire dans `/content`.

- Si le repo est public : aucune information supplémentaire n’est nécessaire.
- S’il est privé : crée un Secret Colab optionnel `GITHUB_TOKEN` avec un token GitHub ayant seulement l’accès lecture au contenu du repo.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = "https://github.com/MilFhey/fit-outfit-advisor.git"
REPO_DIR = Path("/content/fit-outfit-advisor")
BRANCH = "main"


def get_optional_secret(name: str) -> str | None:
    try:
        value = userdata.get(name)
    except Exception:
        return None
    return value or None


def build_git_environment(github_token: str | None) -> tuple[dict[str, str], Path | None]:
    """Retourne un environnement Git ; utilise GIT_ASKPASS seulement si le repo est privé."""
    env = os.environ.copy()

    if not github_token:
        return env, None

    askpass_file = Path("/tmp/git_askpass.sh")
    askpass_file.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )
    askpass_file.chmod(0o700)

    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = str(askpass_file)
    env["GIT_TERMINAL_PROMPT"] = "0"

    return env, askpass_file


github_token = get_optional_secret("GITHUB_TOKEN")
git_env, askpass_file = build_git_environment(github_token)

try:
    # Supprime uniquement un dossier incomplet provenant d’un clone interrompu.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists():
        print(f"Repo déjà présent, mise à jour : {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], env=git_env, check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
            env=git_env,
            check=True,
        )
    else:
        print(f"Clonage du repo : {REPO_URL}")
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            env=git_env,
            check=True,
        )
finally:
    if askpass_file is not None:
        askpass_file.unlink(missing_ok=True)

# Le repo peut contenir le projet directement à sa racine ou dans un dossier imbriqué.
candidate_project_dirs = [
    REPO_DIR,
    REPO_DIR / "fit-outfit-advisor",
]

PROJECT_DIR = next(
    (candidate for candidate in candidate_project_dirs if (candidate / "src").is_dir()),
    None,
)

if PROJECT_DIR is None:
    repo_contents = [path.name for path in REPO_DIR.iterdir()] if REPO_DIR.exists() else []
    raise FileNotFoundError(
        "Impossible de localiser le dossier src du projet. "
        f"Contenu de {REPO_DIR} : {repo_contents}"
    )

os.chdir(PROJECT_DIR)

print(f"Repo cloné : {REPO_DIR}")
print(f"Projet détecté : {PROJECT_DIR}")
print(f"Répertoire courant : {Path.cwd()}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Installer les dépendances


In [ ]:
requirements_path = PROJECT_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt absent : {requirements_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "kaggle"], check=True)

print("✅ Dépendances installées.")


## 4. Créer les dossiers temporaires Colab


In [ ]:
RUNTIME_ROOT = Path("/content/fit-outfit-runtime")
KAGGLE_DOWNLOAD_DIR = RUNTIME_ROOT / "kaggle_downloads"
CONTENT_DATA_DIR = RUNTIME_ROOT / "data"
CONTENT_ARTIFACT_DIR = RUNTIME_ROOT / "artifacts"

for directory in [RUNTIME_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)


## 5. Charger le secret Kaggle

Ton Secret Colab s’appelle **`KAGGLE_API`**.

Sa valeur doit être uniquement ton token Kaggle moderne, qui commence par `KGAT_`.

Le code le lit sous le nom `KAGGLE_API`, puis le place dans `KAGGLE_API_TOKEN` pour que la commande `kaggle` puisse s’authentifier.


In [ ]:
try:
    kaggle_token = userdata.get("KAGGLE_API")
except Exception as exc:
    raise ValueError(
        "Secret Colab introuvable : crée ou autorise le Secret nommé exactement KAGGLE_API."
    ) from exc

if not kaggle_token or not kaggle_token.startswith("KGAT_"):
    raise ValueError(
        "Le Secret KAGGLE_API est absent ou invalide. "
        "Il doit contenir uniquement un token Kaggle moderne commençant par KGAT_."
    )

# Kaggle CLI attend cette variable d’environnement ; le Secret Colab peut conserver ton nom KAGGLE_API.
os.environ["KAGGLE_API_TOKEN"] = kaggle_token
os.environ["KAGGLE_API"] = kaggle_token

print("✅ Token Kaggle chargé depuis le Secret Colab KAGGLE_API.")


## 6. Tester l’accès à Kaggle


In [ ]:
result = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "clothing fit dataset"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Échec d’authentification ou de connexion Kaggle.\n"
        f"Erreur : {result.stderr}"
    )

print(result.stdout[:2000])


## 7. Télécharger le dataset ModCloth


In [ ]:
KAGGLE_DATASET = "rmisra/clothing-fit-dataset-for-size-recommendation"
FORCE_DOWNLOAD = False

existing_files = [path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file()]

if FORCE_DOWNLOAD or not existing_files:
    command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET,
        "-p",
        str(KAGGLE_DOWNLOAD_DIR),
        "--unzip",
    ]
    subprocess.run(command, check=True)
else:
    print("Téléchargement déjà présent : réutilisation des fichiers existants.")

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file())

if not downloaded_files:
    raise FileNotFoundError(
        "Aucun fichier téléchargé depuis Kaggle. Vérifie le token et le slug du dataset."
    )

for path in downloaded_files:
    print(path)


## 8. Détecter le fichier ModCloth

Le dataset peut être fourni en CSV, JSON ou JSONL.  
S’il est en JSON/JSONL, le notebook génère un CSV temporaire dans l’espace Colab.


In [ ]:
import pandas as pd

csv_files = sorted(KAGGLE_DOWNLOAD_DIR.rglob("*.csv"))
modcloth_csv_files = [path for path in csv_files if "modcloth" in path.name.lower()]

if modcloth_csv_files:
    DATASET_PATH = modcloth_csv_files[0]
    print(f"CSV ModCloth détecté : {DATASET_PATH}")
elif csv_files:
    DATASET_PATH = csv_files[0]
    print(f"CSV détecté : {DATASET_PATH}")
else:
    json_files = sorted(
        [*KAGGLE_DOWNLOAD_DIR.rglob("*.json"), *KAGGLE_DOWNLOAD_DIR.rglob("*.jsonl")]
    )
    modcloth_json_files = [path for path in json_files if "modcloth" in path.name.lower()]

    if not modcloth_json_files:
        raise FileNotFoundError(
            "Aucun CSV, JSON ou JSONL ModCloth détecté dans le téléchargement Kaggle."
        )

    source_json = modcloth_json_files[0]
    print(f"JSON/JSONL ModCloth détecté : {source_json}")

    try:
        df_json = pd.read_json(source_json, lines=True)
    except ValueError:
        df_json = pd.read_json(source_json)

    DATASET_PATH = CONTENT_DATA_DIR / "modcloth_final_data.csv"
    df_json.to_csv(DATASET_PATH, index=False)
    print(f"CSV temporaire généré : {DATASET_PATH}")

print(f"DATASET_PATH = {DATASET_PATH}")


## 9. Inspecter le dataset avant entraînement


In [ ]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

print("df.shape =", df.shape)

print("\nColonnes :")
print(list(df.columns))

print("\nTypes :")
display(df.dtypes.to_frame("dtype"))

print("\nAperçu :")
display(df.head())

missing_values = df.isna().sum().sort_values(ascending=False)
print("\nValeurs manquantes par colonne :")
display(missing_values.to_frame("missing_count"))


In [ ]:
TOTAL = len(df)

missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / TOTAL * 100).round(2)
display(missing.sort_values("missing_count", ascending=False))

print("Distribution fit")
display(df["fit"].value_counts(dropna=False))
display((df["fit"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("Distribution category")
display(df["category"].value_counts(dropna=False))
display((df["category"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("Fit par category")
display(pd.crosstab(df["category"], df["fit"]))
display(pd.crosstab(df["category"], df["fit"], normalize="index").round(3))

In [ ]:
explicit_categories = ["tops", "dresses", "bottoms", "outerwear", "wedding"]
ambiguous_categories = ["new", "sale"]

df_no_commercial = df[~df["category"].isin(ambiguous_categories)].copy()
df_explicit = df[df["category"].isin(explicit_categories)].copy()

print("Dataset complet:", df.shape)
print("Sans new/sale:", df_no_commercial.shape)
print("Categories explicites seulement:", df_explicit.shape)

print("Fit complet")
display(df["fit"].value_counts(normalize=True).round(3))

print("Fit sans new/sale")
display(df_no_commercial["fit"].value_counts(normalize=True).round(3))

print("Fit categories explicites")
display(df_explicit["fit"].value_counts(normalize=True).round(3))

In [ ]:
print("size describe")
display(df["size"].describe())

print("size par fit")
display(df.groupby("fit")["size"].describe())

print("size par category")
display(df.groupby("category")["size"].describe())

print("Exemples size/category/fit")
display(df[["size", "category", "fit", "height", "hips", "bra size", "cup size"]].head(30))

In [ ]:
measurement_cols = ["height", "hips", "bra size", "cup size", "waist", "bust"]

for col in measurement_cols:
    print(f"\n=== {col} ===")
    print("missing:", df[col].isna().sum(), f"({df[col].isna().mean() * 100:.2f}%)")
    display(df[col].value_counts(dropna=False).head(30))

print("\nMensurations manquantes par fit")
for col in measurement_cols:
    missing_by_fit = df.assign(is_missing=df[col].isna()).groupby("fit")["is_missing"].mean().round(3)
    print(f"\n{col}")
    display(missing_by_fit)

print("\nMensurations numériques par fit")
for col in ["hips", "bra size", "waist"]:
    print(f"\n{col}")
    display(df.groupby("fit")[col].describe())

print("\nCup size par fit")
display(pd.crosstab(df["cup size"], df["fit"], normalize="index").round(3))

In [ ]:
# Parse simple de height en cm pour analyse descriptive
import re
import numpy as np
import pandas as pd

def parse_height_to_cm(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return np.nan
    if "ft" in text:
        feet = float(nums[0])
        inches = float(nums[1]) if len(nums) > 1 else 0
        return round((feet * 12 + inches) * 2.54, 2)
    if "cm" in text:
        return float(nums[0])
    return np.nan

analysis = df.copy()
analysis["height_cm"] = analysis["height"].map(parse_height_to_cm)

print("height_cm par fit")
display(analysis.groupby("fit")["height_cm"].describe())

print("size par fit")
display(analysis.groupby("fit")["size"].describe())

print("size par category et fit")
display(analysis.groupby(["category", "fit"])["size"].describe())

print("Correlation numérique simple")
numeric_cols = ["size", "height_cm", "hips", "bra size"]
display(analysis[numeric_cols].corr(numeric_only=True))

In [ ]:
candidate_features = ["size", "category", "height", "hips", "bra size", "cup size", "fit"]

subsets = {
    "minimal_size_category_height": ["size", "category", "height", "fit"],
    "with_hips": ["size", "category", "height", "hips", "fit"],
    "with_bra_cup": ["size", "category", "height", "bra size", "cup size", "fit"],
    "with_hips_bra_cup": ["size", "category", "height", "hips", "bra size", "cup size", "fit"],
}

for name, cols in subsets.items():
    temp = df[cols].dropna()
    print(f"\n=== {name} ===")
    print("rows:", len(temp), f"({len(temp) / len(df) * 100:.2f}%)")
    print("fit distribution")
    display(temp["fit"].value_counts(normalize=True).round(3))
    print("category distribution")
    display(temp["category"].value_counts(normalize=True).round(3))

In [ ]:
for col in ["size", "category", "cup size", "bra size", "hips", "height"]:
    print(f"\n=== {col} ===")
    print("nunique:", df[col].nunique(dropna=False))
    display(df[col].value_counts(dropna=False).head(50))

In [ ]:
analysis = df.copy()
analysis["height_cm"] = analysis["height"].map(parse_height_to_cm)

print("height_cm outliers bas")
display(analysis[analysis["height_cm"] < 130][["height", "height_cm", "fit", "category", "size"]].head(50))

print("height_cm outliers hauts")
display(analysis[analysis["height_cm"] > 210][["height", "height_cm", "fit", "category", "size"]].head(50))

print("Nombre outliers height")
print("height_cm < 130:", (analysis["height_cm"] < 130).sum())
print("height_cm > 210:", (analysis["height_cm"] > 210).sum())

In [ ]:
for col in ["size", "hips", "bra size"]:
    print(f"\n=== {col} ===")
    display(df[col].describe())
    display(df[[col, "fit", "category"]].sort_values(col).head(20))
    display(df[[col, "fit", "category"]].sort_values(col, ascending=False).head(20))

In [ ]:
explicit_categories = ["tops", "dresses", "bottoms", "outerwear", "wedding"]
ambiguous_categories = ["new", "sale"]

print("Catégories inconnues")
print(set(df["category"].dropna().unique()) - set(explicit_categories) - set(ambiguous_categories))

print("Counts catégories explicites")
display(df[df["category"].isin(explicit_categories)]["category"].value_counts())

print("Counts new/sale")
display(df[df["category"].isin(ambiguous_categories)]["category"].value_counts())

## 10. Lancer l'entrainement ModCloth V3 - scope all

Cette cellule lance le script experimental `src.training.train_fit_model_v3` sur toutes les categories ModCloth, y compris `new` et `sale`.

Les artefacts sont ecrits dans `models/fit_v3/` avec :
- `model_status: "experimental_only"` ;
- `promotable_to_streamlit: false`.

Ne copie pas ces artefacts vers `models/fit_active/`.


In [ ]:
EPOCHS = 30
BATCH_SIZE = 64

training_script = PROJECT_DIR / "src" / "training" / "train_fit_model_v3.py"

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f"Dataset absent : {DATASET_PATH}")

if not training_script.exists():
    raise FileNotFoundError(f"Script d'entrainement absent : {training_script}")

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    f"{PROJECT_DIR}{os.pathsep}{training_env.get('PYTHONPATH', '')}".rstrip(os.pathsep)
)

print(f"Repertoire courant : {PROJECT_DIR}")
print(f"Script execute : {training_script}")
print(f"Dataset : {DATASET_PATH}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_fit_model_v3",
        "--dataset",
        str(DATASET_PATH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)


## 11. Verifier et sauvegarder le run V3 all

Cette cellule verifie les artefacts du dernier run puis copie le run `all` dans Google Drive.


In [ ]:
import json
import shutil
from pathlib import Path

FIT_V3_DIR = PROJECT_DIR / "models" / "fit_v3"
DRIVE_V3_ALL_DIR = DRIVE_ROOT / "fit-outfit-advisor" / "artifacts" / "modcloth_fit_v3_all"

artifact_paths = [
    FIT_V3_DIR / "fit_model.keras",
    FIT_V3_DIR / "fit_estimator.joblib",
    FIT_V3_DIR / "fit_preprocessor.joblib",
    FIT_V3_DIR / "fit_label_encoder.joblib",
    FIT_V3_DIR / "metadata.json",
    FIT_V3_DIR / "metrics.json",
    FIT_V3_DIR / "confusion_matrix_raw.png",
    FIT_V3_DIR / "confusion_matrix_normalized.png",
    FIT_V3_DIR / "training_history.png",
]

print("Artefacts V3 all :")
for artifact_path in artifact_paths:
    if artifact_path.exists():
        print("OK     ", artifact_path.name, artifact_path.stat().st_size, "bytes")
    else:
        print("ABSENT ", artifact_path.name)

metadata = json.loads((FIT_V3_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((FIT_V3_DIR / "metrics.json").read_text(encoding="utf-8"))

if metadata.get("category_scope") != "all":
    raise RuntimeError(f"Le dernier run n'est pas all : {metadata.get('category_scope')}")

print("\n=== Metadata all ===")
print("version:", metadata.get("version"))
print("category_scope:", metadata.get("category_scope"))
print("selected_experiment:", metadata.get("selected_experiment"))
print("selected_model_type:", metadata.get("selected_model_type"))
print("model_status:", metadata.get("model_status"))
print("promotable_to_streamlit:", metadata.get("promotable_to_streamlit"))
print("reason:", metadata.get("reason_for_selection"))

DRIVE_V3_ALL_DIR.mkdir(parents=True, exist_ok=True)
for artifact_path in FIT_V3_DIR.iterdir():
    if artifact_path.is_file() and artifact_path.name != ".gitkeep":
        shutil.copy2(artifact_path, DRIVE_V3_ALL_DIR / artifact_path.name)
        print("copied", artifact_path.name)

print("Saved all run to:", DRIVE_V3_ALL_DIR)


## 12. Lancer l'entrainement ModCloth V3 - categories explicites

Cette cellule relance V3 en excluant les categories commerciales `new` et `sale`.

Le run ecrase temporairement `models/fit_v3/`, puis la cellule suivante le copie dans un dossier Drive separe.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_fit_model_v3",
        "--dataset",
        str(DATASET_PATH),
        "--category-scope",
        "explicit",
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)


## 13. Verifier et sauvegarder le run V3 explicit

Cette cellule verifie les artefacts du dernier run puis copie le run `explicit` dans Google Drive.


In [ ]:
FIT_V3_DIR = PROJECT_DIR / "models" / "fit_v3"
DRIVE_V3_EXPLICIT_DIR = DRIVE_ROOT / "fit-outfit-advisor" / "artifacts" / "modcloth_fit_v3_explicit"

print("Artefacts V3 explicit :")
for artifact_path in artifact_paths:
    if artifact_path.exists():
        print("OK     ", artifact_path.name, artifact_path.stat().st_size, "bytes")
    else:
        print("ABSENT ", artifact_path.name)

metadata = json.loads((FIT_V3_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((FIT_V3_DIR / "metrics.json").read_text(encoding="utf-8"))

if metadata.get("category_scope") != "explicit":
    raise RuntimeError(f"Le dernier run n'est pas explicit : {metadata.get('category_scope')}")

print("\n=== Metadata explicit ===")
print("version:", metadata.get("version"))
print("category_scope:", metadata.get("category_scope"))
print("selected_experiment:", metadata.get("selected_experiment"))
print("selected_model_type:", metadata.get("selected_model_type"))
print("model_status:", metadata.get("model_status"))
print("promotable_to_streamlit:", metadata.get("promotable_to_streamlit"))
print("reason:", metadata.get("reason_for_selection"))

DRIVE_V3_EXPLICIT_DIR.mkdir(parents=True, exist_ok=True)
for artifact_path in FIT_V3_DIR.iterdir():
    if artifact_path.is_file() and artifact_path.name != ".gitkeep":
        shutil.copy2(artifact_path, DRIVE_V3_EXPLICIT_DIR / artifact_path.name)
        print("copied", artifact_path.name)

print("Saved explicit run to:", DRIVE_V3_EXPLICIT_DIR)


## 14. Resume des sauvegardes Drive

Cette cellule confirme que les deux runs V3 sont presents dans Google Drive.


In [ ]:
for drive_dir in [DRIVE_V3_ALL_DIR, DRIVE_V3_EXPLICIT_DIR]:
    print("\n", drive_dir)
    if not drive_dir.exists():
        print("ABSENT")
        continue
    for artifact_path in sorted(drive_dir.iterdir()):
        if artifact_path.is_file():
            print(artifact_path.name, artifact_path.stat().st_size, "bytes")
